# Perplexity Experimentation for Different Models

I want to explore the differences in perplexity across different models across different datasets. This will setup some of my work with perplexity for token prediction, etc. I will be testing on these models
- GPT2
- Llama3-8B
- Llama2-7B
- Mistral-7B
- Gemma-7B
- OpenChat3.6-8B

In [1]:
!pip install -qU torch  
!pip install -qU transformers
!pip install -qU datasets altair huggingface_hub tqdm accelerate python-dotenv
!pip install -qU sentencepiece

import torch as t
from torch.nn import functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import altair as alt
from huggingface_hub import notebook_login
from tqdm import tqdm

import os
from dotenv import load_dotenv

device = "cuda" if t.cuda.is_available() else "cpu"


[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
stdout = sys.stdout

In [3]:
!nvidia-smi

Wed Jun  5 03:27:09 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:00:0C.0 Off |                    0 |
| N/A   34C    P0             54W /  300W |       3MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
HF_TOKEN = os.getenv("HF_TOKEN")
notebook_login()

In [5]:
GPT2_MODEL = "openai-community/gpt2-medium"
MISTRAL_MODEL = "mistralai/Mistral-7B-v0.3"
LLAMA3_MODEL = "meta-llama/Meta-Llama-3-8B"
LLAMA2_MODEL = "meta-llama/Llama-2-7b-hf"
GEMMA_MODEL = "google/gemma-7b"
OPENCHAT_MODEL = "openchat/openchat-3.6-8b-20240522"

## Datasets 

I will be using 3 different datasets to test out 
- Salesforce/wikitext
- THUDM/humaneval-x
- meta-math/MetaMathQA

We will evaluate all the models on a dataset by dataset basis. Over here I am evaluating the `Salesforce/wikitext` dataset. Head over to 
- [human_eval.ipynb](./human_eval.iynb) for the HumanEval dataset
- [mathqa.ipynb](./mathqa.ipynb) for the meta-math/MetaMathQA dataset

### Wikitext Evals

We will run all the models on wikitext and figure out the perplexities. We first get the **entire dataset** and compare the model's outputs to the actual text.

In [ ]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
# I'm only taking the first 500 out because otherwise its too slow
data = "\n".join(dataset["text"][:100])
print(f"The number of words in the wikitext dataset are {len(data.split(' '))}")

wikitext_ppls = dict() 

In [ ]:
def calculate_perplexity(model, tokenized_dataset, context_len, sequence_len, stride=512):
    nlls = []
    prev_end_loc = 0
    for start_loc in tqdm(range(0, sequence_len, stride)):
        end_loc = start_loc + context_len
        tgt_loc = end_loc - prev_end_loc

        # tokenize the inputs 
        input_ids = tokenized_dataset.input_ids[:, start_loc:end_loc]
        
        input_ids = input_ids.to(device)
        target_ids = input_ids.clone() 
        target_ids[:,:-tgt_loc] = -100 

        # calculate the model's loss
        with t.no_grad():
            outputs = model(input_ids, labels=target_ids)
            nll = outputs.loss
            nlls.append(nll)

        prev_end_loc = end_loc 
        if end_loc == sequence_len:
            break
    return t.exp(t.tensor(nlls).mean())

**GPT2 Evals**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL)

In [ ]:
context_len = model.config.n_positions
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {GPT2_MODEL} is {ppl:.2f}")
wikitext_ppls[GPT2_MODEL] = ppl.item()

**Mistral Evals**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MISTRAL_MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL)

In [ ]:
!nvidia-smi

In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {MISTRAL_MODEL} is {ppl:.2f}")
wikitext_ppls[MISTRAL_MODEL] = ppl.item()

**Llama3 Evals**

In [8]:
model = AutoModelForCausalLM.from_pretrained(LLAMA3_MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(LLAMA3_MODEL)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [9]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {LLAMA3_MODEL} is {ppl:.2f}")
wikitext_ppls[LLAMA3_MODEL] = ppl.item()

NameError: name 'data' is not defined

**Llama2 Evals**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(LLAMA2_MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(LLAMA2_MODEL)

In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {LLAMA2_MODEL} is {ppl:.2f}")
wikitext_ppls[LLAMA2_MODEL] = ppl.item()

**Gemma 7B Evals**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(GEMMA_MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL)

In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {GEMMA_MODEL} is {ppl:.2f}")
wikitext_ppls[GEMMA_MODEL] = ppl.item()

**OpenChat-3.6 8B Evals**

In [6]:
model = AutoModelForCausalLM.from_pretrained(OPENCHAT_MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(OPENCHAT_MODEL)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {OPENCHAT_MODEL} is {ppl:.2f}")
wikitext_ppls[OPENCHAT_MODEL] = ppl.item()

In [ ]:
wikitext_ppls

In [9]:
dataset = load_dataset("THUDM/humaneval-x", split="test", trust_remote_code=True)
# I'm only taking the first 500 out because otherwise its too slow
prompts = dataset["prompt"]
print(prompts[:10])

humaneval_ppls = dict()

['from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    """ Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    """\n', 'from typing import List\n\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to\n    separate those group into separate strings and return the list of those.\n    Separate groups are balanced (each open brace is properly closed) and not nested within each other\n    Ignore any spaces in the input string.\n    >>> separate_paren_groups(\'( ) (( )) (( )( ))\')\n    [\'()\', \'(())\', \'(()())\']\n    """\n', '\n\ndef truncate_number(number: float) -> float:\n    """ Given a positive floating point number, it can b

In [10]:
from transformers.generation.logits_process import TopKLogitsWarper, TopPLogitsWarper

def top_k_top_p_filtering(
    logits: t.FloatTensor,
    top_k: int = 0,
    top_p: float = 1.0,
    filter_value: float = -float("Inf"),
    min_tokens_to_keep: int = 1,
) -> t.FloatTensor:

    if top_k > 0:
        logits = TopKLogitsWarper(top_k=top_k, filter_value=filter_value, min_tokens_to_keep=min_tokens_to_keep)(
            None, logits
        )

    if 0 <= top_p <= 1.0:
        logits = TopPLogitsWarper(top_p=top_p, filter_value=filter_value, min_tokens_to_keep=min_tokens_to_keep)(
            None, logits
        )

    return logits


def calculate_mean_nll_loss(model, tokenizer, prompt):

    num_sampled = 0
    nll_vals = []    
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    while True:
        outputs = model(input_ids)
        # Get logits from last layer
        last_layer_logits = outputs.logits[:, -1, :]
        # Keep top 100 logits at max; stop if cumulative probability >= 1.0.
        top_logits = top_k_top_p_filtering(last_layer_logits, top_k=10, top_p=1.0)
        probs = F.softmax(top_logits, dim=-1)
        token_idx = t.argmax(probs, dim=-1)
        probability = probs[0,token_idx]
        generated_next_token = t.multinomial(probs, num_samples=1)
        nll_vals.append(-1 * t.log(probability))
        print(generated_next_token.item(), token_idx.item(), generated_next_token == tokenizer.eos_token_id)

        # Once the model is done predicting, we calculate the perplexity by taking the 
        # exp of the mean nll values.
        if generated_next_token == tokenizer.eos_token_id:
            print("Ended")
            ppl = t.exp(t.stack(nll_vals, dim=0).mean())
            return ppl
        
        input_ids = t.cat([input_ids, generated_next_token], dim=-1)
        num_sampled += 1

In [12]:
# out = calculate_mean_nll_loss(model, tokenizer, prompts[0])
output = calculate_mean_nll_loss(model, tokenizer, prompts[0])

262 262 tensor([[False]], device='cuda:0')
369 5219 tensor([[False]], device='cuda:0')
602 602 tensor([[False]], device='cuda:0')
304 304 tensor([[False]], device='cuda:0')
2134 2134 tensor([[False]], device='cuda:0')
7 7046 tensor([[False]], device='cuda:0')
15 16 tensor([[False]], device='cuda:0')
11 11 tensor([[False]], device='cuda:0')
2479 2479 tensor([[False]], device='cuda:0')
48307 48307 tensor([[False]], device='cuda:0')
8 8 tensor([[False]], device='cuda:0')
482 482 tensor([[False]], device='cuda:0')
220 220 tensor([[False]], device='cuda:0')
16 16 tensor([[False]], device='cuda:0')
997 997 tensor([[False]], device='cuda:0')
286 286 tensor([[False]], device='cuda:0')
422 369 tensor([[False]], device='cuda:0')
3731 3731 tensor([[False]], device='cuda:0')
48307 48307 tensor([[False]], device='cuda:0')
1004 1004 tensor([[False]], device='cuda:0')
10 60 tensor([[False]], device='cuda:0')
16 16 tensor([[False]], device='cuda:0')
60 60 tensor([[False]], device='cuda:0')
482 482 ten

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 

In [11]:
!nvidia-smi

Wed Jun  5 03:28:35 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:00:0C.0 Off |                    0 |
| N/A   34C    P0             74W /  300W |   31057MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----